# 62 — Structured Dynamical Systems (DyS)

**Dataset:** Structured Retrospective (task-planning app; N=61 behavioral participants)
**Model family:** Dynamical Systems (DyS)
**Blueprint:** `outputs/reports/structured/structured_dys_blueprint.md` — read this first. NOT a port of
DS5's DyS notebook (DS5 reduced its own 1D StoP SDE to a sigma->0 deterministic limit). The theory
paper's own DyS section describes a multi-dimensional WEEKLY state-space trajectory instead, and does
not depend on StoP's fitted daily parameters.
**Output root:** `outputs/model_outputs/structured/dynamical_systems/`

---

## The weekly state vector (5 dimensions, all named in the paper's own Example A)

`n_created_w, timed_share_w, inbox_share_w, completed_share_w, complexity_w` — computed over tasks
CREATED in week `w` (participant-relative week index, local calendar day via `event_timezone`).
Restricted to non-censored tasks (`has_creation_event=True`) — a real limitation, carried forward via
`pct_tasks_left_censored` in the output table, not hidden.

`completed_share_w` = fraction of that week's created tasks EVER later completed (eventual fate, not
same-week completion).


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
OUT = PROJECT_ROOT / "outputs" / "model_outputs" / "structured" / "dynamical_systems"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

MIN_WEEKS = 4

print("Output folder:", OUT)

## 1. Load Data

In [ ]:
beh = pd.read_csv(CLEANED / "structured_behavioral_clean.csv")
beh["event_timestamp_utc"] = pd.to_datetime(beh["event_timestamp_utc"], format="mixed", utc=True)
tasks = pd.read_csv(FEATURES / "structured_task_lifecycle_features.csv")
pf = pd.read_csv(FEATURES / "structured_participant_features.csv")

print(f"Tasks: {len(tasks):,}  Participants: {pf.shape[0]}")

## 2. Local-Day Conversion for Task Created Events (same convention as StoP notebook 63)

In [ ]:
def to_local_date_series(df):
    out = pd.Series(index=df.index, dtype="object")
    for tz, grp in df.groupby("event_timezone"):
        try:
            out.loc[grp.index] = grp["event_timestamp_utc"].dt.tz_convert(tz).dt.date
        except Exception:
            out.loc[grp.index] = pd.NaT
    return out

created = beh[beh["event_type"] == "Task Created"].copy()
created["local_date"] = to_local_date_series(created)
created = created.dropna(subset=["local_date"])
created["local_date"] = pd.to_datetime(created["local_date"])
print(f"Non-censored creation events: {len(created):,}")

## 3. Assign Participant-Relative Week Index

In [ ]:
first_creation = created.groupby("participant_id")["local_date"].transform("min")
created["week_idx"] = ((created["local_date"] - first_creation).dt.days // 7).astype(int)

weeks_per_p = created.groupby("participant_id")["week_idx"].nunique()
print("distinct weeks with creation activity per participant:")
print(weeks_per_p.describe())
print(f"\nEligible (>= {MIN_WEEKS} weeks): {(weeks_per_p >= MIN_WEEKS).sum()} / {len(weeks_per_p)}")

## 4. Build Weekly State Vectors

In [ ]:
task_status = tasks.set_index("task_id")["lifecycle_status"]
created["eventually_completed"] = created["task_id"].map(task_status).eq("completed")
created["subtasks_amount"] = created["task_id"].map(tasks.set_index("task_id")["subtasks_amount_last"])

def week_vector(grp):
    n = len(grp)
    type_counts = grp["task_type"].value_counts(normalize=True)
    return pd.Series({
        "n_created_w": n,
        "timed_share_w": type_counts.get("timed", 0.0),
        "inbox_share_w": type_counts.get("inbox", 0.0),
        "all_day_share_w": type_counts.get("all_day", 0.0),
        "completed_share_w": grp["eventually_completed"].mean(),
        "complexity_w": grp["subtasks_amount"].mean(),
    })

weekly = created.groupby(["participant_id", "week_idx"]).apply(week_vector).reset_index()
print(f"Weekly state vectors: {weekly.shape}")
weekly.head(8)

## 5. Save Weekly State Vectors (Long Format)

In [ ]:
weekly.to_csv(OUT / "weekly_state_vectors_long.csv", index=False)
print(f"Saved: weekly_state_vectors_long.csv  ({weekly.shape})")

## 6. Trajectory Metrics per Participant

In [ ]:
STATE_COLS = ["n_created_w", "timed_share_w", "inbox_share_w", "completed_share_w", "complexity_w"]

# z-score each state dimension across the whole eligible-participant population for volatility calc
eligible_pids = weeks_per_p[weeks_per_p >= MIN_WEEKS].index
weekly_elig = weekly[weekly["participant_id"].isin(eligible_pids)].copy()
z = weekly_elig[STATE_COLS].copy()
z = (z - z.mean()) / z.std(ddof=0).replace(0, np.nan)
weekly_elig[[c + "_z" for c in STATE_COLS]] = z

records = []
for pid, grp in weekly.groupby("participant_id"):
    n_weeks = grp["week_idx"].nunique()
    is_eligible = n_weeks >= MIN_WEEKS
    rec = {"participant_id": pid, "n_weeks": n_weeks, "is_dys_eligible": is_eligible}

    for c in STATE_COLS:
        rec[f"centroid_{c}"] = round(float(grp[c].mean()), 4)

    if is_eligible:
        g = grp.sort_values("week_idx")
        gz = weekly_elig[weekly_elig["participant_id"] == pid].sort_values("week_idx")
        zcols = [c + "_z" for c in STATE_COLS]
        diffs = gz[zcols].diff().dropna()
        volatility = float(np.sqrt((diffs ** 2).sum(axis=1)).mean()) if len(diffs) > 0 else np.nan
        rec["state_space_volatility"] = round(volatility, 4)

        # volume trend slope (OLS)
        x = g["week_idx"].values.astype(float)
        y = g["n_created_w"].values.astype(float)
        if len(x) >= 2 and x.std() > 0:
            slope = float(np.polyfit(x, y, 1)[0])
        else:
            slope = np.nan
        rec["volume_trend_slope"] = round(slope, 4)

        # dropout score: (last quartile mean - first quartile mean) / overall mean
        n = len(g)
        q = max(1, n // 4)
        first_q_mean = g["n_created_w"].iloc[:q].mean()
        last_q_mean = g["n_created_w"].iloc[-q:].mean()
        overall_mean = g["n_created_w"].mean()
        rec["dropout_score"] = round(float((last_q_mean - first_q_mean) / overall_mean), 4) if overall_mean > 0 else np.nan
    else:
        rec["state_space_volatility"] = np.nan
        rec["volume_trend_slope"] = np.nan
        rec["dropout_score"] = np.nan

    records.append(rec)

traj_df = pd.DataFrame(records)

# Participants with ZERO non-censored creation events never appear in `created`/`weekly` at all --
# add them back explicitly as ineligible rows so eligibility percentages are correctly out of all 61
# participants, not silently out of only the 57 who have >=1 non-censored task.
all_pids = set(pf["participant_id"].unique())
missing_pids = all_pids - set(traj_df["participant_id"])
print(f"Participants with zero non-censored creation events (100% left-censored): {len(missing_pids)}")
if missing_pids:
    fill_cols = ["n_weeks"] + [f"centroid_{c}" for c in STATE_COLS] +                 ["state_space_volatility", "volume_trend_slope", "dropout_score"]
    missing_rows = pd.DataFrame([
        {"participant_id": pid, "is_dys_eligible": False, **{c: np.nan for c in fill_cols}}
        for pid in missing_pids
    ])
    traj_df = pd.concat([traj_df, missing_rows], ignore_index=True)

traj_df = traj_df.merge(pf[["participant_id", "pct_tasks_left_censored"]], on="participant_id", how="left")

elig = traj_df[traj_df["is_dys_eligible"]]
print(f"Eligible participants: {len(elig)} / {len(traj_df)} ({len(elig)/len(traj_df)*100:.1f}%)")
print(elig[["state_space_volatility", "volume_trend_slope", "dropout_score"]].describe().round(3))

## 7. Descriptive Region Labels (Part 5 of blueprint — rule-based, NOT clustering)

In [ ]:
def assign_labels(elig):
    dropout_terc = elig["dropout_score"].quantile([1/3, 2/3]).values
    vol_terc = elig["state_space_volatility"].quantile([1/3, 2/3]).values
    volume_terc = elig["centroid_n_created_w"].quantile([1/3, 2/3]).values
    timed_terc = elig["centroid_timed_share_w"].quantile([1/3, 2/3]).values
    completed_terc = elig["centroid_completed_share_w"].quantile([1/3, 2/3]).values
    inbox_terc = elig["centroid_inbox_share_w"].quantile([1/3, 2/3]).values

    def classify(r):
        if r["dropout_score"] <= dropout_terc[0]:
            return "dropout-prone"
        if r["state_space_volatility"] >= vol_terc[1]:
            return "oscillatory"
        if r["centroid_n_created_w"] <= volume_terc[0] and r["state_space_volatility"] <= vol_terc[0]:
            return "sparse-consistent"
        if r["centroid_timed_share_w"] >= timed_terc[1] and r["centroid_completed_share_w"] >= completed_terc[1]:
            return "high-structure"
        if r["centroid_inbox_share_w"] >= inbox_terc[1]:
            return "inbox-heavy"
        return "mixed"

    return elig.apply(classify, axis=1)

traj_df.loc[elig.index, "region_label"] = assign_labels(elig)
print(traj_df["region_label"].value_counts(dropna=False))

## 8. Subscription Group Comparison (observational)

In [ ]:
sub_last = beh.sort_values("event_timestamp_utc").groupby("participant_id")["subscription_type_at_event"].last()
traj_df["subscription_type_last"] = traj_df["participant_id"].map(sub_last)

sub_summary = traj_df[traj_df["is_dys_eligible"]].groupby("subscription_type_last")[
    ["state_space_volatility", "volume_trend_slope", "dropout_score"]
].agg(["mean", "median", "count"])
print(sub_summary.round(3))
print()
print("NOTE: purely observational, no causal claim.")

## 9. Save Trajectory Metrics

In [ ]:
traj_df.to_csv(OUT / "participant_trajectory_metrics.csv", index=False)
sub_summary.to_csv(OUT / "subscription_group_summary.csv")
print("Saved: participant_trajectory_metrics.csv, subscription_group_summary.csv")

## 10. Left-Censoring Caveat Check (Part 2 of blueprint)

In [ ]:
print("pct_tasks_left_censored among DyS-eligible participants:")
print(elig_check := traj_df[traj_df["is_dys_eligible"]]["pct_tasks_left_censored"].describe().round(3))
print()
print("Correlation(state_space_volatility, pct_tasks_left_censored):",
      round(traj_df[traj_df["is_dys_eligible"]][["state_space_volatility","pct_tasks_left_censored"]].corr().iloc[0,1], 3))
print("If strongly correlated, high-censoring participants' trajectories may be artificially volatile")
print("(thin, unrepresentative slice of their true task portfolio) rather than genuinely oscillatory.")

## 11. Figure 1 — PCA Projection of Weekly Trajectories

In [ ]:
pca_input = weekly_elig[[c + "_z" for c in STATE_COLS]].dropna()
pca = PCA(n_components=2)
coords = pca.fit_transform(pca_input)
plot_df = weekly_elig.loc[pca_input.index].copy()
plot_df["pc1"] = coords[:, 0]
plot_df["pc2"] = coords[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap("tab20")
for i, (pid, g) in enumerate(plot_df.groupby("participant_id")):
    g = g.sort_values("week_idx")
    ax.plot(g["pc1"], g["pc2"], marker="o", markersize=3, alpha=0.5, color=cmap(i % 20), linewidth=1)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Weekly state-space trajectories (PCA projection, one line per participant)")
plt.tight_layout()
plt.savefig(FIG / "fig1_dys_state_space_pca.png", dpi=150)
plt.show()
print("Saved fig1_dys_state_space_pca.png")
print(f"Variance explained by PC1+PC2: {pca.explained_variance_ratio_.sum()*100:.1f}%")

## 12. Figure 2 — Region Label Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
traj_df["region_label"].value_counts(dropna=False).plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Number of participants")
ax.set_title("DyS descriptive region labels (Step 1, rule-based)")
plt.tight_layout()
plt.savefig(FIG / "fig2_dys_region_label_distribution.png", dpi=150)
plt.show()
print("Saved fig2_dys_region_label_distribution.png")

## 13. Figure 3 — Volatility vs. Dropout Score

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = {"dropout-prone": "#C44E52", "oscillatory": "#DD8452", "sparse-consistent": "#4C72B0",
          "high-structure": "#55A868", "inbox-heavy": "#8172B2", "mixed": "#999999"}
for label, g in elig.assign(region_label=traj_df.loc[elig.index, "region_label"]).groupby("region_label"):
    ax.scatter(g["dropout_score"], g["state_space_volatility"], label=label,
               color=colors.get(label, "black"), s=50, alpha=0.8)
ax.set_xlabel("dropout_score (negative = declining activity)")
ax.set_ylabel("state_space_volatility")
ax.legend(fontsize=8)
ax.axvline(0, color="gray", lw=0.8, ls="--")
plt.tight_layout()
plt.savefig(FIG / "fig3_dys_volatility_vs_dropout.png", dpi=150)
plt.show()
print("Saved fig3_dys_volatility_vs_dropout.png")

## 14. model_run_summary.md

In [ ]:
summary = f"""# Structured DyS -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/62_structured_dynamical_systems.ipynb
**Blueprint:** outputs/reports/structured/structured_dys_blueprint.md

## Eligibility
- DyS-eligible (>= {MIN_WEEKS} weeks with creation activity): {int(traj_df['is_dys_eligible'].sum())}/{len(traj_df)} ({traj_df['is_dys_eligible'].mean()*100:.1f}%) -- out of all 61 participants
  (of whom {len(missing_pids)} have zero non-censored creation events and are therefore automatically ineligible)

## Trajectory metrics (eligible, N={len(elig)})
{elig[["state_space_volatility","volume_trend_slope","dropout_score"]].describe().round(3).to_string()}

## Region label distribution
{traj_df["region_label"].value_counts(dropna=False).to_string()}

## Left-censoring caveat check
Correlation(state_space_volatility, pct_tasks_left_censored) = {round(traj_df[traj_df["is_dys_eligible"]][["state_space_volatility","pct_tasks_left_censored"]].corr().iloc[0,1], 3)}

## Subscription group comparison (observational only, no causal claim)
{sub_summary.round(3).to_string()}

## PCA of weekly state space
Variance explained by PC1+PC2: {pca.explained_variance_ratio_.sum()*100:.1f}%
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)